# The eight problems, run live

Companion to `ISSUES.md`. Every section states the problem, then calls the real
library function and shows it going wrong. Nothing is stubbed or monkeypatched.

Everything here uses hand-built tables of a few rows, so it runs in seconds and
needs **no CellOracle object**. Run it on the `fftemporal` kernel.

| # | problem | what you see below |
| --- | --- | --- |
| 1 | "Both TFs strongly regulate this gene" always returns nothing | 4 genes expected, 0 returned |
| 2 | One thin cluster kills the whole run | `ValueError` from pandas |
| 3 | Downstream-gene list wrapped in an extra list | `len` says 1, truth is 5 |
| 4 | Order-2 tables sorted by scores they no longer contain | `0.0` sorted above `2.0` |
| 5 | `slide_starting_genes` too low empties results silently | p-value `nan`, row vanishes |
| 6 | Impossible overlap counts accepted | `p = 0.0` from impossible input |
| 7 | Score and p-value contradict each other | "no enrichment" + "highly significant" |
| 8 | Cluster names have to be numbers | `ValueError` on `'ActB_1'` |

## Setup

In [1]:
import itertools

import pandas as pd
from scipy.stats import hypergeom

from focal.state_specific._slide import (
    build_enrichment_table,
    get_slide_grn_enrichment,
    hypergeom_slide_grn_score,
)
from focal.state_specific._grn import filter_network_scores_for_clusters
from focal.state_specific import StateSpecificEnrichment

CF = ('c0',)   # a one-cluster "fusion", the key cc_dict is stored under


def edges(rows):
    """Build an edges_df from (source, target, weight, strength) tuples."""
    return pd.DataFrame(
        [{'source': s, 'target': t, 'key': 0, 'weight': w, 'strength': st}
         for s, t, w, st in rows]
    )


def run(edges_df, ord_tf, slide, sources, universe=100):
    cc = {}
    get_slide_grn_enrichment(edges_df, cc, CF, ord_tf, slide, universe,
                             sources, 'slide', show_progress=False)
    return {tuple(r[1]): r for r in cc[CF][ord_tf]}   # keyed by strength condition

---
## 1. Asking for "both TFs strongly regulate this gene" always returns nothing

When you score a *pair* of TFs you also pick a strength pattern: should TF1's edge
be strong or weak, and TF2's? Four patterns exist — `(0,0)`, `(0,1)`, `(1,0)`,
`(1,1)`.

Two of them, `(1,1)` and `(0,0)`, never return a single gene no matter what the
data says.

Below, TFA and TFB **both strongly regulate G1, G2, G3 and G4**. G2 and G3 are
SLIDE genes. The right answer for `(1,1)` is four downstream genes and two shared
ones. Watch it return nothing.

In [2]:
co_regulated = edges([(tf, g, 1.0, 1)
                      for tf in ('TFA', 'TFB')
                      for g in ('G1', 'G2', 'G3', 'G4')])
co_regulated

,source,target,key,weight,strength
0,TFA,G1,0,1.0,1
1,TFA,G2,0,1.0,1
2,TFA,G3,0,1.0,1
3,TFA,G4,0,1.0,1
4,TFB,G1,0,1.0,1
5,TFB,G2,0,1.0,1
6,TFB,G3,0,1.0,1
7,TFB,G4,0,1.0,1


In [3]:
res = run(co_regulated, ord_tf=2, slide={'G2', 'G3'}, sources={'TFA', 'TFB'})

for condition, rec in sorted(res.items()):
    shared, downstream = rec[3]
    print(f'pattern {condition}:  downstream={downstream}  shared={shared}')

print()
print('pattern (1,1) is the strongest possible signal in this network.')
print('It returned', len(res[(1, 1)][3][1]), 'downstream genes. Expected 4.')

pattern (0, 0):  downstream=[]  shared=[]
pattern (0, 1):  downstream=[]  shared=[]
pattern (1, 0):  downstream=[]  shared=[]
pattern (1, 1):  downstream=[]  shared=[]

pattern (1,1) is the strongest possible signal in this network.
It returned 0 downstream genes. Expected 4.


### Why

A gene survives only if this is true:

```python
filtered.groupby("target")[["source", "strength"]].nunique().eq(len(filter_df)).all(axis=1)
```

`len(filter_df)` is 2 (two TFs). The check demands two distinct `source` values —
correct — **and two distinct `strength` values**. Under `(1,1)` every row has
strength 1, so there is one distinct value, not two, and the gene is dropped.

Here is that exact check run by hand:

In [4]:
tf_comb, condition = ('TFA', 'TFB'), (1, 1)

filter_df = pd.DataFrame({'source': tf_comb, 'strength': condition})
filtered  = co_regulated.merge(filter_df, on=['source', 'strength'])

counts = filtered.groupby('target')[['source', 'strength']].nunique()
print('distinct values per gene:')
print(counts)
print()
print(f'compared against len(filter_df) = {len(filter_df)}')
print()
print('passes?')
print(counts.eq(len(filter_df)))
print()
print('-> source column is right; strength column can never be 2. Every gene dropped.')

distinct values per gene:
        source  strength
target                  
G1           2         1
G2           2         1
G3           2         1
G4           2         1

compared against len(filter_df) = 2

passes?
        source  strength
target                  
G1        True     False
G2        True     False
G3        True     False
G4        True     False

-> source column is right; strength column can never be 2. Every gene dropped.


In [5]:
# A MIXED pattern passes the same check, which is what isolates the bug to the
# homogeneous case rather than to the check as a whole.
mixed = edges([('TFA', g, 1.0, 1) for g in ('G2', 'G3')] +
              [('TFB', g, 0.1, 0) for g in ('G2', 'G3')])

res_mixed = run(mixed, ord_tf=2, slide={'G2', 'G3'}, sources={'TFA', 'TFB'})
print('pattern (1,0) downstream genes:', sorted(res_mixed[(1, 0)][3][1]))
print('-> works fine')

pattern (1,0) downstream genes: ['G2', 'G3']
-> works fine


### Why it matters

`build_enrichment_table` uses `[(1,1), (0,1), (1,0)]` as the default pattern list
for order-2 runs — so the default includes a pattern that can never return
anything, and it is the one most people would ask for first.

Order 1 is unaffected: one TF, `len(filter_df)` is 1, one strength value, check
passes.

In [6]:
rec = run(co_regulated, ord_tf=1, slide={'G2', 'G3'}, sources={'TFA'})[(1,)]
print('order 1, TFA, strong edges ->', sorted(rec[3][1]))
print('-> order 1 is fine')

order 1, TFA, strong edges -> ['G1', 'G2', 'G3', 'G4']
-> order 1 is fine


---
## 2. One thin cluster kills the whole run

If a cluster yields fewer **candidate TFs** than the order you asked for,
`run_enrichment` stops with a pandas error and everything already computed is lost.

### Which TFs are candidates

`run_enrichment` builds the list like this:

```python
slide_in_graph      = slide_features & set(grn.nodes)
neighbours          = every direct regulator (predecessor) of those genes
slide_tf_candidates = (slide_in_graph | neighbours) & GRN_TFs
```

A gene is a candidate when it is a known TF (listed in `GRN_TFs`) **and** either
it directly regulates a SLIDE gene in this cluster's thresholded network, or it is
itself a SLIDE gene appearing in that network.

The network is rebuilt per cluster fusion *after* quantile thresholding, so the
list is recomputed each time and can be very short for a sparse cluster.

In [7]:
# Who counts as a candidate, worked through.
from focal.state_specific._grn import combine_cluster_grn_links, grn_edges_from_combined_links

demo_links = {0: pd.DataFrame({
    'source':    ['TFA', 'TFB', 'TFC'],
    'target':    ['S1',  'X1',  'X2'],
    'coef_mean': [1.0, 1.0, 1.0],
    'coef_abs':  [1.0, 1.0, 1.0]})}

combined, _ = combine_cluster_grn_links(demo_links, (0,), quantile=0.5)
grn, _      = grn_edges_from_combined_links(combined)

slide_features_demo = {'S1', 'TFC', 'G9'}        # G9 is not in the network
GRN_TFs_demo        = ['TFA', 'TFB', 'TFC', 'TFD']   # TFD is not in the network

slide_in_graph = slide_features_demo & set(grn.nodes)
neighbours     = {r for g in slide_in_graph for r in grn.predecessors(g)}
candidates     = (slide_in_graph | neighbours) & set(GRN_TFs_demo)

print('network nodes   :', sorted(grn.nodes))
print('SLIDE in network:', sorted(slide_in_graph), '  <- G9 dropped, not a node')
print('their regulators:', sorted(neighbours))
print('candidates      :', sorted(candidates))
print()
for tf in GRN_TFs_demo:
    if tf in neighbours:       why = 'candidate  - regulates a SLIDE gene'
    elif tf in slide_in_graph: why = 'candidate  - is itself a SLIDE gene in the network'
    elif tf not in grn.nodes:  why = 'excluded   - not in the network'
    else:                      why = 'excluded   - in the network, regulates no SLIDE gene'
    print(f'  {tf}: {why}')

network nodes   : ['S1', 'TFA', 'TFB', 'TFC', 'X1', 'X2']
SLIDE in network: ['S1', 'TFC']   <- G9 dropped, not a node
their regulators: ['TFA']
candidates      : ['TFA', 'TFC']

  TFA: candidate  - regulates a SLIDE gene
  TFB: excluded   - in the network, regulates no SLIDE gene
  TFC: candidate  - is itself a SLIDE gene in the network
  TFD: excluded   - not in the network


Below there is exactly **one** candidate TF and we ask for pairs.

In [8]:
links = {0: pd.DataFrame({'source': ['TFA', 'TFA'],
                          'target': ['G1', 'G2'],
                          'coef_mean': [2.0, 1.0],
                          'coef_abs':  [2.0, 1.0]})}


def build(tmpdir, order_fr_tfcomb, grn_tfs):
    sse = StateSpecificEnrichment(
        grn_workdir=tmpdir, oracle_object_name='unused', feature_folder=tmpdir,
        out_path=tmpdir, slide_starting_genes=100, clusters_of_interest=[0],
        order_fr_clust=[1], order_fr_tfcomb=order_fr_tfcomb, quantile=0.5,
    )
    sse.GRN_links_after_fit = {k: v.copy() for k, v in links.items()}
    sse.GRN_TFs = grn_tfs
    sse.slide_features = {'G2'}
    return sse

In [9]:
import tempfile
from pathlib import Path

sse = build(Path(tempfile.mkdtemp()), order_fr_tfcomb=[2], grn_tfs=['TFA'])

try:
    sse.run_enrichment(show_progress=False, save_strength_diagnostics=False)
    print('completed')
except ValueError as exc:
    print(f'{type(exc).__name__}: {exc}')
    print()
    print('One usable TF, pairs requested -> no combinations -> empty results list.')
    print('The message points at pandas, not at the real cause.')

ValueError: Columns must be same length as key

One usable TF, pairs requested -> no combinations -> empty results list.
The message points at pandas, not at the real cause.


In [10]:
# Same crash from the other direction: no TF survives the GRN_TFs intersection.
sse = build(Path(tempfile.mkdtemp()), order_fr_tfcomb=[1], grn_tfs=['TF_NOT_IN_GRN'])

try:
    sse.run_enrichment(show_progress=False, save_strength_diagnostics=False)
    print('completed')
except ValueError as exc:
    print(f'{type(exc).__name__}: {exc}')

ValueError: Columns must be same length as key


In [11]:
# The root cause, isolated: an empty results list cannot be turned into a table.
for order in (1, 2):
    try:
        build_enrichment_table({CF: {order: []}}, CF, order)
        print(f'order {order}: ok')
    except ValueError as exc:
        print(f'order {order}: {type(exc).__name__}: {exc}')

print()
print('An empty table would be the sensible outcome, not an exception.')

order 1: ValueError: Columns must be same length as key
order 2: ValueError: Columns must be same length as key

An empty table would be the sensible outcome, not an exception.


In [12]:
# For contrast: one usable TF and order 1 requested works.
sse = build(Path(tempfile.mkdtemp()), order_fr_tfcomb=[1], grn_tfs=['TFA'])
sse.run_enrichment(show_progress=False, save_strength_diagnostics=False)
print('completed, records:', len(sse._records((0,), 1)))

completed, records: 2


---
## 3. The downstream-gene list is wrapped in an extra list

When a TF has downstream genes but **none of them are SLIDE genes**, the gene list
is recorded one level too deep. `len()` then reports 1 instead of the real count.

In [13]:
five = edges([('TFX', f'D{i}', 1.0, 1) for i in range(5)])

no_overlap = run(five, ord_tf=1, slide={'ZZZ'}, sources={'TFX'})[(1,)]
overlap    = run(five, ord_tf=1, slide={'D0'},  sources={'TFX'})[(1,)]

print('no SLIDE gene downstream:')
print('   value :', no_overlap[3][1])
print('   len   :', len(no_overlap[3][1]), '   <- wrong, there are 5 genes')
print()
print('one SLIDE gene downstream (every other branch):')
print('   value :', sorted(overlap[3][1]))
print('   len   :', len(overlap[3][1]), '   <- correct')

no SLIDE gene downstream:
   value : [['D0', 'D1', 'D2', 'D3', 'D4']]
   len   : 1    <- wrong, there are 5 genes

one SLIDE gene downstream (every other branch):
   value : ['D0', 'D1', 'D2', 'D3', 'D4']
   len   : 5    <- correct


### Why it matters

At order 1 these rows are dropped anyway (score is 0, the filter needs > 0), so it
never shows. At order 2 failing rows are **kept** with their score zeroed, so the
malformed list is written into `enriched_df_*.csv` and anything reading that column
back gets a nested list. The "more than 2 downstream genes" filter is also counting
the wrong thing.

---
## 4. Order-2 tables are sorted by scores they no longer contain

Rows are sorted by score, and *then* the failing rows have their score set to 0.
So a row showing `0.0` can sit above a row showing `2.0`.

In [14]:
def record(tf, condition, score, p, common, downstream, case='slide'):
    return [tuple(tf), tuple(condition), (score, p),
            (list(common), list(downstream)), case, (pd.DataFrame(), pd.DataFrame())]


passes      = record(('A', 'B'), (1, 1), 2.0, 0.01, ['g1', 'g2'], ['d1', 'd2', 'd3'])
high_but_ns = record(('E', 'F'), (0, 1), 9.0, 0.20, ['g1', 'g2'], ['d1', 'd2', 'd3'])
#                                        ^^^ big score, but p = 0.20 fails the filter

out = build_enrichment_table({CF: {2: [passes, high_but_ns]}}, CF, 2)
out[['TF', 'condition', 'score', 'p_value']]

,TF,condition,score,p_value
1,"(E, F)","(0, 1)",0.0,1.00
0,"(A, B)","(1, 1)",2.0,0.01


In [15]:
print('row order :', [t for t in out['TF']])
print('scores    :', list(out['score']))
print()
print('(E, F) sorts first on its original 9.0, then gets zeroed -- so the file')
print('shows 0.0 above 2.0. The numbers are right; the order is misleading.')

row order : [('E', 'F'), ('A', 'B')]
scores    : [0.0, 2.0]

(E, F) sorts first on its original 9.0, then gets zeroed -- so the file
shows 0.0 above 2.0. The numbers are right; the order is misleading.


---
## 5. Setting `slide_starting_genes` too low silently empties the results

`slide_starting_genes` is the size of the gene universe for the hypergeometric
test. If a TF has more downstream genes than that number, the test is being asked
something impossible — more successes in the population than the population holds.

scipy returns `nan` for the p-value. The score still looks like a normal number.

In [16]:
# universe = 50, but 80 downstream genes
score, p_value = hypergeom_slide_grn_score(50, 80, 10, 5)
print(f'score   = {score}')
print(f'p_value = {p_value}')
print()
print('p_value < 0.05  ->', p_value < 0.05, '  (nan compares False)')
print()
print('The row is dropped at order 1, or zeroed at order 2. No warning either way.')
print('You see an empty results table, which reads like "no enrichment found".')

score   = -1.6780719051126376
p_value = nan

p_value < 0.05  -> False   (nan compares False)

The row is dropped at order 1, or zeroed at order 2. No warning either way.
You see an empty results table, which reads like "no enrichment found".


In [17]:
# A sane universe, same network: the result appears.
score, p_value = hypergeom_slide_grn_score(500, 80, 10, 5)
print(f'universe 500 -> score {score:.4f}, p {p_value:.4g}, passes filter: {p_value < 0.05}')

universe 500 -> score 1.6439, p 0.01213, passes filter: True


A check worth adding at the start of a run: `slide_starting_genes` should be at
least as large as the biggest downstream-gene count in the network.

---
## 6. Impossible overlap counts are accepted without complaint

Two inputs that cannot occur produce confident-looking results instead of an error.

In [18]:
cases = [
    ('more shared genes than SLIDE genes   (s > S)', (100, 20, 5, 8)),
    ('more shared genes than downstream    (s > p)', (100, 3, 10, 5)),
]
for label, args in cases:
    score, p = hypergeom_slide_grn_score(*args)
    print(f'{label}:  score={score:.4f}  p={p:.4g}')

print()
print('Both look like the strongest possible enrichment.')

more shared genes than SLIDE genes   (s > S):  score=3.0000  p=0
more shared genes than downstream    (s > p):  score=4.0589  p=0

Both look like the strongest possible enrichment.


In [19]:
# The other degenerate cases ARE guarded, which is what makes the gap inconsistent.
for label, args in [('no shared genes', (100, 20, 10, 0)),
                    ('empty sample',    (100, 20, 0, 0)),
                    ('empty universe',  (0, 0, 0, 0))]:
    print(f'{label:18} -> {hypergeom_slide_grn_score(*args)}')

no shared genes    -> (0.0, 1.0)
empty sample       -> (0.0, 1.0)
empty universe     -> (0.0, 1.0)


---
## 7. Score and p-value contradict each other when there are no downstream genes

The score has a divide-by-zero guard that returns 0. The p-value has no matching
guard, so scipy computes it anyway.

In [20]:
score, p_value = hypergeom_slide_grn_score(100, 0, 10, 3)
print(f'score   = {score}      -> "no enrichment"')
print(f'p_value = {p_value}      -> "as significant as possible"')
print()
print('Only reachable through bad input, but the two halves disagree.')

score   = 0.0      -> "no enrichment"
p_value = 0.0      -> "as significant as possible"

Only reachable through bad input, but the two halves disagree.


---
## 8. Cluster names have to be numbers

`filter_network_scores_for_clusters` converts each cluster id with `int(cid)`.
Everywhere else accepts any label — `clusters_of_interest` is typed `list[Any]`,
and the rest of the module works with whatever keys the oracle has.

In [21]:
scores_numeric = pd.DataFrame(
    {'degree_centrality_out': [0.9, 0.5, 0.7, 0.2], 'cluster': [0, 0, 1, 1]},
    index=['GA', 'GB', 'GA', 'GC'],
)
filter_network_scores_for_clusters((0, 1), scores_numeric)

,degree_centrality_out,cluster
GA,0.9,0
GB,0.5,0
GC,0.2,1


In [22]:
scores_named = scores_numeric.copy()
scores_named['cluster'] = ['ActB_1', 'ActB_1', 'GC_1', 'GC_1']

try:
    filter_network_scores_for_clusters(('ActB_1',), scores_named)
except ValueError as exc:
    print(f'{type(exc).__name__}: {exc}')
    print()
    print('The B cell oracle uses \'0\'..\'11\', which survive int().')
    print('A named clustering would not.')

ValueError: invalid literal for int() with base 10: 'ActB_1'

The B cell oracle uses '0'..'11', which survive int().
A named clustering would not.


---
## Summary

| # | problem | severity |
| --- | --- | --- |
| 1 | Homogeneous strength patterns return nothing at order ≥ 2 | wrong answers, silently |
| 2 | A thin cluster aborts the whole run | crash, work lost |
| 3 | Downstream list nested when nothing overlaps | malformed order-2 output |
| 4 | Order-2 rows sorted by pre-zeroing scores | misleading, numbers fine |
| 5 | Low `slide_starting_genes` empties results | looks like "no enrichment" |
| 6 | Impossible counts accepted | latent |
| 7 | Score and p-value disagree | latent |
| 8 | Cluster names must be numeric | latent for this dataset |

Each one is also pinned by a test in this folder, so a fix shows up as an `XPASS`:

```bash
conda activate fftemporal
pytest tests/state_specific -q -rx
```